# Credit Risk – Bronze Layer

## Ziel

In diesem Notebook werden die Rohdaten aus `dataset.csv` in den Bronze Layer der Medallion Architecture geladen.

Der Bronze Layer enthält die Daten möglichst unverändert. Es findet hier noch keine Datenbereinigung, Feature-Auswahl oder Modellierung statt.

### Datenquelle

- Datei: `dataset.csv`
- Quelle: Databricks Unity Catalog Volume
- Schema: `Data_Science`
- Volume: `credit_analytics`

### Ziel

Die Rohdaten werden als Delta-Tabelle `credit_risk_bronze` gespeichert.

```text
dataset.csv
   ↓
Bronze Layer
   ↓
Data_Science.credit_risk_bronze





### Wir erstellen das Volume credit_analytics

In [0]:
%sql
create volume if not exists Data_Science.credit_analytics;

In [0]:
### 1. `dataset.csv` einlesen

# Pfad zur Rohdatei im Unity Catalog Volume
source_path ='/Volumes/workspace/Data_Science/credit_analytics/dataset.csv'

# Rohdaten einlesen
df_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

display(df_bronze.limit(1))

id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag
1077501,5000,5000,4975,36 months,10.65%,162.87,B,B2,null,10+ years,RENT,24000,Verified,Dec-2011,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=1077501,Borrower added on 12/22/11 > I need to upgrade my business technologies.,credit_card,Computer,860xx,AZ,27.65,0.0,Jan-1985,735.0,739.0,1.0,3.0,0.0,13648.0,83.7%,9.0,f,0.0,0.0,5863.1551866952,5833.84,5000.0,863.16,0.0,0.0,0.0,Jan-2015,171.62,Dec-2018,749.0,745.0,0.0,1.0,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N


# 2. Rohdaten kontrollieren

In [0]:
# Schema anzeigen
df_bronze.printSchema()

# Anzahl Zeilen und Spalten
print("Anzahl Zeilen:", df_bronze.count())
print("Anzahl Spalten:", len(df_bronze.columns))

root
 |-- id: integer (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- funded_amnt_inv: integer (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: string (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: integer (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- delinq_2yrs: 

## 3. Bronze-Tabelle speichern

In [0]:
bronze_dataset = "Data_Science.credit_risk_bronze"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_dataset)
)

## 4. Kontrolle der Bronze-Tabelle

In [0]:
df_bronze_check = spark.table("Data_Science.credit_risk_bronze")

display(df_bronze_check.limit(1))

id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag
1077501,5000,5000,4975,36 months,10.65%,162.87,B,B2,null,10+ years,RENT,24000,Verified,Dec-2011,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=1077501,Borrower added on 12/22/11 > I need to upgrade my business technologies.,credit_card,Computer,860xx,AZ,27.65,0.0,Jan-1985,735.0,739.0,1.0,3.0,0.0,13648.0,83.7%,9.0,f,0.0,0.0,5863.1551866952,5833.84,5000.0,863.16,0.0,0.0,0.0,Jan-2015,171.62,Dec-2018,749.0,745.0,0.0,1.0,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N


**Einige Prüfungen**

In [0]:
print("Anzahl Zeilen:", df_bronze_check.count())
print("Anzahl Spalten:", len(df_bronze_check.columns))

Anzahl Zeilen: 39717
Anzahl Spalten: 60


## Ergebnis

Die Rohdaten aus `dataset.csv` wurden erfolgreich in den Bronze Layer übernommen.

Die Tabelle

`Data_Science.credit_risk_bronze`

bildet die unveränderte Ausgangsbasis für die weiteren Verarbeitungsschritte.

Die Datenbereinigung und Aufbereitung erfolgt anschließend im **Silver Layer**.


### Abschluss der Bronze-Schicht

Nach dem Einlesen und der technischen Vorbereinigung der `dataset.csv` werden die Daten als Bronze-Tabelle gespeichert.

Die Bronze-Schicht enthält damit die **technisch aufbereiteten Ausgangsdaten**. 
Es erfolgt an dieser Stelle noch keine fachliche Spezialisierung für PD oder LGD.

**Ergebnis:**



```text


dataset.csv
    │
    ▼
Credit_Analytics_Risk_bronze
    │
    ▼
credit_risk_bronze